In [28]:
import numpy as np
import matplotlib.pyplot as plt

from contextuality.measurement_scenario import MeasurementScenario
from contextuality.empirical_model import EmpiricalModel
from contextuality.utils import compute_NCF, compute_max_CF

Scenario:

In [29]:
X = [i for i in range(5)]
M = [[i,i+1] for i in range(4)] + [[4,0]]
O = [0,1]
kcbs = MeasurementScenario(X, M, O)

print(M)

[[0, 1], [1, 2], [2, 3], [3, 4], [4, 0]]


Quanmtum realization as empirical model

In [30]:
meas = np.zeros((5,2,3,3))# shape = number mesurements, number of outcomes, dimension of state (d x d)
N = 1/np.sqrt(1+np.cos(np.pi/5))
for i in range(5):
    vec = N * np.array([np.cos(4*np.pi*i/5), np.sin(4*np.pi*i/5), np.sqrt(np.cos(np.pi/5))])
    meas[i][1] = np.outer(vec, vec)
    meas[i][0] = np.eye(3) - meas[i][1]

psi = np.array([0,0,1])
rho = np.outer(psi, psi)

In [31]:
empirical_model = EmpiricalModel(kcbs)
empirical_model.quantum_realisation(rho, meas)

In [ ]:
KCBS_bound = 0
C = np.zeros(empirical_model.vector.shape)
for i in range(5):
    KCBS_bound += empirical_model.vector[1 + 4 * i]
    C[1+4*i] = 1

# Quantum Bound (Lovasz): np.sqrt(5) = 2.23606797749979
KCBS_bound

Contextual Fraction

In [ ]:
result = compute_NCF(empirical_model, solver='MOSEK', verbose=False)
CF = result['CF']
CF

In [ ]:
# Get the maximum contextual fraction
result_OD_NS = compute_max_CF(kcbs, sigma=0, eta=0)
print(result_OD_NS["EmpiricalModel"].vector)
result_CF = compute_NCF(result_OD_NS["EmpiricalModel"])
print(result_CF['CF'])

In [ ]:
# When there is 1-eta OD
result_OD_NS = compute_max_CF(kcbs, sigma=0, eta=0.3)
print(result_OD_NS["EmpiricalModel"].vector.reshape(4, 5))
result_CF = compute_NCF(result_OD_NS["EmpiricalModel"])
print(result_CF['CF'])

Contextual Fraction Bound

In [ ]:
eta = np.linspace(0, 1, 2)
sigma = np.linspace(0, 1, 2)

CF = np.zeros((eta.size, sigma.size))

for i, _eta in enumerate(eta):
    for j, _sigma in enumerate(sigma):
        res = compute_max_CF(kcbs, _sigma, _eta)
        CF_res = compute_NCF(res["EmpiricalModel"])
        CF[i, j] = CF_res["CF"]

In [ ]:
fig, ax = plt.subplots(figsize=(12,6))

cs = ax.contourf(eta, sigma, CF, zorder=5, cmap=plt.cm.coolwarm)
ax.xaxis.grid(True, zorder=0)
plt.xticks(fontsize=14, rotation=0)
ax.yaxis.grid(True, zorder=0)
plt.yticks(fontsize=14, rotation=0)
plt.xlabel('$\eta$', fontsize=16)
plt.ylabel('$\sigma$', fontsize=16)
plt.xlim(0, 1)
plt.ylim(0, 1)
fig.colorbar(cs, ax=ax, shrink=0.9)